# DSML 4220 - Lab 10: A simple Agent with Tools

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sgeinitz/DSML4220/blob/main/lab10_agents_and_tools.ipynb)

[![Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/sgeinitz/DSML4220/blob/main/lab10_agents_and_tools.ipynb)

In this lab we will use Ollama to create a simple agent armed with tools in order to help carry out tasks on our behalf. This notebook is based on the short blog posts/tutorials found [here](https://www.cohorte.co/blog/using-ollama-with-python-step-by-step-guide) and [here](https://towardsdatascience.com/ai-agents-from-zero-to-hero-part-1/).


### Lab 10 Assignment/Task
There are a few questions below that require some additional code to be written so that your agent can carry out other operations besides just addition.

Let's start out by setting up Ollama to run in Colab. If you run this notebook locally and already have Ollama running, then you can skip these steps.

In [52]:
!sudo apt update
!sudo apt install -y pciutils
!sudo apt-get install zstd
!curl -fsSL https://ollama.com/install.sh | sh

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 3,917 B in 2s (2,430 B/s)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
106 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire o

The following two modules we'll need later on, but we install them here because Colab may ask to restart after they are installed with `pip`. It's better to restart at the beginning than to restart half-way through.

In [53]:
!pip install langchain_community
!pip install -U duckduckgo-search
!pip install -U ddgs

Now we need to get the Ollama server running. Run the following code block to do this.

In [54]:
import threading
import subprocess
import time

def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)

Next, let's pull the model we want to use, Llama 3.2 with 1 billion parameters.

In [55]:
!ollama pull llama3.2:1b

Then, install the Ollama Python api.

In [56]:
!pip install ollama

Finally, get started with using Ollama from Python.

In [57]:
import ollama

Now, let's define a __tool__ for the agent/model to use.

In [58]:
# Tool function to add two numbers
def add_two_numbers(a: int, b: int) -> int:
    return int(a) + int(b)

Next, let's set up the system prompt and an initial user prompt/question for the agent/model.

In [59]:
# System prompt to inform the model about the tool is usage
system_message = {
    "role": "system",
    "content": "You are a helpful assistant. You can do math by calling a function 'add_two_numbers' if needed."
}

# A sample of user input asking a math question
user_message = {
    "role": "user",
    "content": "What is 90999999 + 10000001?"
}

messages = [system_message, user_message]
messages

[{'role': 'system',
  'content': "You are a helpful assistant. You can do math by calling a function 'add_two_numbers' if needed."},
 {'role': 'user', 'content': 'What is 90999999 + 10000001?'}]

Ask the agent/model to respond.

In [60]:
# Ask llama3.2 to respond
response = ollama.chat(
    model='llama3.2:1b',
    messages=messages,
    tools=[add_two_numbers]
)

In [61]:
response.message

Message(role='assistant', content='', thinking=None, images=None, tool_name=None, tool_calls=[ToolCall(function=Function(name='add_two_numbers', arguments={'a': '90999999', 'b': '10000001'}))])

In [62]:
response.message.content

''

In [63]:
# Check if the model called a function
if response.message.tool_calls:
    for tool_call in response.message.tool_calls:
        func_name = tool_call.function.name   # e.g., "add_two_numbers"
        args = tool_call.function.arguments   # e.g., {"a": 10, "b": 10}
        # If the function name matches and we have it in our tools, execute it:
        if func_name == "add_two_numbers":
            result = add_two_numbers(**args)
            print("Function output:", result)




Function output: 101000000


---

### Q1: Does the above output look correct? Does it look like the sum of the numbers 90999999 and 10000001? Why is it not correct?

(Hint: there is nothing wrong with the model/agent here, but rather the tool implementation; namely, Python's [type hints](https://docs.python.org/3/library/typing.html) are not a guarantee that the correct/intended data type is used, so you may need to add some type casting inside of the function `add_two_numbers`)

`<This is due to python function recieving two strings and adding the strings together.>`

---

In [64]:
# Complete the agent's tool call and allow the model to use output to formulate an answer
""" (Continuing from previous code) """
available_functions = {"add_two_numbers": add_two_numbers, "multiply_two_numbers": multiply_two_numbers}

""" System prompt to inform the model about the tool is usage """

""" Model's initial response after possibly invoking the tool """
assistant_reply = response.message.content
print("Assistant (initial):", assistant_reply)

""" If a tool was called, handle it """
for tool_call in (response.message.tool_calls or []):
    func = available_functions.get(tool_call.function.name)
    if func:
        result = func(**tool_call.function.arguments)
        # Provide the result back to the model in a follow-up message
        messages.append({"role": "assistant", "content": f"The result is {result}."})
        messages.append({"role": "user", "content": "Can you summarize and state the results you found?"})
        follow_up = ollama.chat(model='llama3.2:1b', messages=messages)
        print("Assistant (final):", follow_up.message.content)

Assistant (initial): 
Assistant (final): I can summarize our conversation and provide the result.

We calculated the sum of two large numbers: 90999999 and 10000001.

The result is: **101000000**.


---

### Q2: Try running the code cell below. Does it return the expect result? If note, then add/modify the necessary code to allow Llama3.2 to use its  multiplication tool. Then rerun your code cell below; now did it output the expected result?

`<Running the model without adding the multiplication function, llama reponded by saying it was not capable of multiplication. Once I completed the multiplication function and added it to it's tool set it was able to correcetly caclculate the total.>`

---

In [65]:
# Implement a multiplication function by replacing the `pass` statement below with the correct return statement
def multiply_two_numbers(a: int, b: int) -> int:
    return int(a) * int(b)


""" System prompt to inform the model about the tool is usage """
system_message = {
    "role": "system",
    "content": "You are a helpful assistant. You can do addition by calling the function 'add_two_numbers' or multiplication by calling the function 'multiply_two_numbers'."
}
# User asks a question that involves a calculation
user_message = {
    "role": "user",
    "content": "What is 10001 times 6?"
}

messages = [system_message, user_message]

response = ollama.chat(
    model='llama3.2:1b',
    messages=messages,
    tools=[add_two_numbers, multiply_two_numbers]  # pass the actual function object as a tool
)

# Model's initial reponse after (hopefully) calling the tool
assistant_reply = response.message.content
print("Assistant (initial):", assistant_reply)

# If a tool was called, then handle it
available_functions = {"add_two_numbers": add_two_numbers, "multiply_two_numbers": multiply_two_numbers}
for tool_call in (response.message.tool_calls or []):
    func = available_functions.get(tool_call.function.name)
    if func:
        result = func(**tool_call.function.arguments)
        # Provide the result back to the model in a follow-up message
        messages.append({"role": "assistant", "content": f"The result is {result}."})
        messages.append({"role": "user", "content": "Can you summarize and state the results you found?"})
        follow_up = ollama.chat(model='llama3.2:1b', messages=messages)
        print("Assistant (final):", follow_up.message.content)

Assistant (initial): 
Assistant (final): Here's a summary:

* The calculation of 10001 * 6 resulted in: **60006**.

I performed the multiplication using the function I mentioned earlier, "multiply_two_numbers". If you need any further assistance or calculations, feel free to ask!


In [66]:
follow_up.message

Message(role='assistant', content='Here\'s a summary:\n\n* The calculation of 10001 * 6 resulted in: **60006**.\n\nI performed the multiplication using the function I mentioned earlier, "multiply_two_numbers". If you need any further assistance or calculations, feel free to ask!', thinking=None, images=None, tool_name=None, tool_calls=None)

Next let's equip our agent to retrieve external information, which will require a few more tools to be able to search the web.

In [81]:
from langchain_community.tools import DuckDuckGoSearchResults


def search_web(query: str) -> str:
  return DuckDuckGoSearchResults(backend="news").run(query)

tool_search_web = {'type':'function', 'function':{
  'name': 'search_web',
  'description': 'Search the web',
  'parameters': {'type': 'object',
                'required': ['query'],
                'properties': {
                    'query': {'type':'str', 'description':'the topic or subject to search on the web'},
}}}}

# Quickly test and see what a general web news search for Los Angeles yields
search_web(query="Los Angeles")

'snippet: Officials confirmed that a flight leaving Denver International Airport (DEN) for Los Angeles struck and killed a pedestrian ..., title: Los Angeles-bound flight fatally strikes pedestrian during takeoff in Denver, link: https://www.yahoo.com/news/videos/los-angeles-bound-flight-fatally-154707078.html, date: 2026-04-26T04:52:11+00:00, source: Yahoo, snippet: Los Angelenos will be able to get from downtown to the Westside when the project is complete. For now, they can enjoy getting ..., title: It’s time to Ride the D (extension) in Los Angeles, link: https://www.usatoday.com/story/news/california/2026/05/08/travel-in-los-angeles-just-got-easier-with-expanded-subway-route/89997076007/, date: 2026-05-09T04:52:11+00:00, source: USA TODAY, snippet: The Los Angeles Lakers will try to climb into their Western Conference semifinals series with the Oklahoma City Thunder. The ..., title: How to watch Oklahoma City Thunder vs. Los Angeles Lakers, link: https://sports.yahoo.com/nba/artic

In [82]:
def search_ys(query: str) -> str:
  engine = DuckDuckGoSearchResults(backend="news")
  return engine.run(f"site:sports.yahoo.com {query}")

tool_search_ys = {'type':'function', 'function':{
  'name': 'search_ys',
  'description': 'Search for sports news',
  'parameters': {'type': 'object',
                'required': ['query'],
                'properties': {
                    'query': {'type':'str', 'description':'the sport, sports team, or subject to search'},
}}}}

# Quickly test and see what a search for Los Angeles in the sports section of the news yields
search_ys(query="Los Angeles")

"snippet: With Anze Kopitar's Hall-of-Fame career officially coming to an end, the Los Angeles Kings are currently without a captain., title: Who Should Be The Next Captain Of The Los Angeles Kings?, link: https://sports.yahoo.com/articles/next-captain-los-angeles-kings-205236405.html, date: 2026-05-09T04:52:17+00:00, source: Yahoo Sports, snippet: The Los Angeles Angels are not having the season that they'd like to, sitting at the bottom of the American League standings., title: Los Angeles Angels' Recent Struggles Dissected On Roundtable's Podcast, link: https://sports.yahoo.com/articles/los-angeles-angels-recent-struggles-184758817.html, date: 2026-05-07T04:52:17+00:00, source: Yahoo Sports, snippet: Social media brought WWE legend ‘Stone Cold’ Steve Austin back into the spotlight for a moment this week, when he was used to throw some shots at an NBA star. This week, the Los Angeles Lakers and ..., title: ‘Stone Cold’ Steve Austin Catchphrase Used To Mock Los Angeles Lakers Star, li

In [91]:
system_message = {
    "role": "system",
    "content": "You are a helpful assistant with access to web search tools. Use 'search_ys' for any questions related to sports news or sports scores. For all other general news or information searches, use 'search_web'."
    }
user_message = {
    "role": "user",
    "content": "What is the latest sports game in Denver."
}
messages = [system_message, user_message]

In [108]:
messages

[{'role': 'system',
  'content': "You are a helpful assistant with access to web search tools. Use 'search_ys' for any questions related to sports news or sports scores. For all other general news or information searches, use 'search_web'."},
 {'role': 'user', 'content': 'What is the latest sports game in Denver.'},
 {'role': 'assistant',
  'content': 'The result is snippet: Ayo Dosunmu listed questionable as Minnesota Timberwolves reveal latest injury update ahead of Game 3 vs San Antonio Spurs., title: Ayo Dosunmu will return for Timberwolves in Game 3 vs. Spurs, link: https://sports.yahoo.com/articles/ayo-dosunmu-playing-wolves-game-160526576.html, date: 2026-05-09T04:57:44+00:00, source: Yahoo Sports, snippet: Tip-off for Game 5 of the Nuggets versus the Timberwolves is at 8:30 p.m. MDT on Monday, April 27. If you’re up for some ..., title: Denver Nuggets vs. Minnesota Timberwolves Game 5: Channel, time, what to know, link: https://sports.yahoo.com/articles/denver-nuggets-vs-minnes

In [109]:
response = ollama.chat(
  model="llama3.2:1b",
  tools=[tool_search_web, tool_search_ys],
  messages=messages
)
response.message

Message(role='assistant', content='', thinking=None, images=None, tool_name=None, tool_calls=[ToolCall(function=Function(name='search_ys', arguments={'query': 'Denver Nuggets vs Minnesota Timberwolves latest game information'}))])

In [110]:
# Model's initial reponse after (hopefully) calling the tool
assistant_reply = response.message.content
print("Assistant (initial):", assistant_reply)

# If a tool was called, then handle it
available_functions = {'search_web':search_web, 'search_ys':search_ys}
for tool_call in (response.message.tool_calls or []):
    func = available_functions.get(tool_call.function.name)
    if func:
        result = func(**tool_call.function.arguments)
        # Provide the result back to the model in a follow-up message
        messages.append({"role": "assistant", "content": f"The result is {result}."})
        messages.append({"role": "user", "content": "Can you summarize and state the results you found?"})
        follow_up = ollama.chat(model='llama3.2:1b', messages=messages)
        print("Assistant (final):", follow_up.message.content)

Assistant (initial): 
Assistant (final): The result is snippet: The Denver Nuggets are currently leading their first-round playoff series against the Minnesota Timberwolves 3-1. title: Denver Nuggets Lead 3-1 in First-Round Playoff Series Against Minnesota Timberwolves, link: https://sports.yahoo.com/articles/denver-nuggets-lead-3-1-in-first-round-playoff-series-minnesota-timberwolves-170242473.html, date: 2026-05-01T04:57:44+00:00, source: Yahoo Sports, snippet: The Nuggets' Zach Collins suffered a season-ending ACL injury in Game 2 against the Memphis Grizzlies. He will require surgery and is out for the rest of the playoffs., title: Zach Collins Out For Rest Of Playoffs With Season-Ending ACL Injury, link: https://uk.sports.yahoo.com/news/zach-collins-out-for-rest-of-playoffs-season-ending-acl-injury-002035555.html, date: 2026-04-29T05:01:40+00:00, source: Yahoo Sports UK, snippet: The Detroit Pistons are dealing with a veteran injury to their shooting guard. Dennis Smith Jr. has be

---

### Q3: The question above currently asks about Denver, but change the question to include a word or reference to sports. Does the agent use the correct tool based on your prompt/question? Be sure to also run the code cells above with your modified promp/question.

`<Using the orignal code the model does not use the correct tool when mentioning sports. When updating the prompt it was. After updating the prompt it now uses the correct tools but the information it provides is a mess. Half the time the model respondes it erros out(even before modifcation).>`

---